# PyTorch basics

Every neural network, from a 3-layer classifier to a 400B-parameter LLM, is built from three things you'll learn here: tensors (n-dimensional arrays whose shapes you have to track constantly), autograd (PyTorch records the operations you do and computes gradients automatically), and the training loop (forward pass, loss, backward pass, update, repeat). You'll also see what gradient descent is doing geometrically, why real training uses noisy mini-batches, and what the optimizers you'll use for the rest of the course actually compute. The notebook ends with a small neural network learning `sin(x)` on your Mac's GPU.

### In this notebook

1. Creating tensors
2. Shapes: indexing, reshaping, permuting
3. Broadcasting
4. Matrix multiplication
5. Using your Mac's GPU
6. Autograd
7. Gradient descent by hand
8. The same thing, the PyTorch way
9. The geometry of gradient descent
10. Local minima, saddle points, and noisy gradients
11. Momentum, Adam, and AdamW
12. A real (small) neural network

Sections 1 to 5 are about tensors, 6 to 8 build the training loop, 9 to 11 explain what the loop is doing to the loss and how optimizers improve on it, and 12 puts it all together. Exercises and solutions are at the bottom.

### Before you start

- Finish module 00 first. `matplotlib` comes from there (`uv add matplotlib`), so this module installs nothing new.
- No apps need to be running and nothing downloads. Plan on 2 to 3 hours to work through it. Every cell runs in seconds.
- Pick the `.venv` kernel in `ML_Lab` (top right of the notebook in VS Code).
- If the linear algebra feels shaky, watch the first four episodes of 3Blue1Brown's "Essence of Linear Algebra" first. Matrix multiplication as "transforming space" is the intuition you want, and it makes sections 4 and 9 easier.

Run cells in order. Before you run a cell that prints a shape, predict the shape first. Shape intuition is the skill that separates people who can debug models from people who can't.

In [ ]:
import time
import torch

print("PyTorch", torch.__version__)

# Pick the fastest device available. "mps" is Apple's GPU backend. "cuda" is NVIDIA (you'd see it on a rented GPU).
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

_ = torch.manual_seed(42)  # makes random numbers repeatable

## 1. Creating tensors

A tensor is like a NumPy array with two extra powers: it can live on a GPU, and it can track gradients.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])           # from a Python list
b = torch.zeros(2, 3)                        # 2 rows, 3 columns of zeros
c = torch.ones(2, 3)
d = torch.randn(2, 3)                        # samples from a standard normal distribution
e = torch.arange(0, 10)                      # 0..9
f = torch.linspace(0, 1, steps=5)            # 5 evenly spaced points from 0 to 1

for name, t in [("a", a), ("b", b), ("d", d), ("e", e), ("f", f)]:
    print(f"{name}: shape={tuple(t.shape)} dtype={t.dtype}\n{t}\n")

Notice `e` is `int64` while the others are `float32`. Neural networks train in floating point, and mixing dtypes is a common source of errors. You'll also see `float16` and `bfloat16` later: half-precision formats that LLMs use to cut memory in half.

In [ ]:
x = torch.arange(6)
print(x.dtype, "->", x.float().dtype, "->", x.to(torch.bfloat16).dtype)

# Bytes per element tells you memory cost. A 7B model in float32 vs bfloat16:
for dt in [torch.float32, torch.bfloat16, torch.int8]:
    bytes_per = torch.tensor([], dtype=dt).element_size()
    print(f"{str(dt):16} {bytes_per} bytes/param -> 7B params = {7e9 * bytes_per / 1e9:.0f} GB")

That last table is the same math behind quantization in module 01.

## 2. Shapes: indexing, reshaping, permuting

Think of a shape as nested boxes. Shape `(2, 3, 4)` means 2 blocks, each with 3 rows, each with 4 numbers. In LLM code you'll constantly see `(batch, sequence_length, hidden_dim)`, for example `(32, 1024, 4096)`.

In [ ]:
t = torch.arange(24).reshape(2, 3, 4)
print(t)
print("shape:", t.shape)
print()
print("t[0]        ->", t[0].shape)         # first block
print("t[0, 1]     ->", t[0, 1].shape)      # first block, second row
print("t[:, 0, :]  ->", t[:, 0, :].shape)   # first row of every block
print("t[..., -1]  ->", t[..., -1].shape)   # last column everywhere ("..." means all leading dims)

In [ ]:
t = torch.arange(24).reshape(2, 3, 4)

print(t.reshape(6, 4).shape)        # same 24 numbers, new shape
print(t.reshape(-1).shape)          # -1 means "figure this dim out" -> flatten
print(t.reshape(2, -1).shape)       # keep batch dim, flatten the rest

print(t.permute(2, 0, 1).shape)     # reorder dimensions: new order is old dims (2, 0, 1)
print(t.transpose(1, 2).shape)      # swap two dims

print(t.unsqueeze(0).shape)         # add a size-1 dim at the front
print(t.unsqueeze(0).squeeze(0).shape)

`reshape` keeps the numbers in the same order and just regroups them. `permute` and `transpose` actually reorder which axis is which. Mixing those two up silently scrambles your data, and it's one of the most common bugs in model code.

## 3. Broadcasting

When you combine tensors of different shapes, PyTorch stretches size-1 dimensions to match, without copying memory. The rule: line the shapes up from the right. Each pair of dims must be equal, or one of them must be 1 (or missing).

In [ ]:
M = torch.ones(4, 3)
row = torch.tensor([10.0, 20.0, 30.0])        # shape (3,)  -> treated as (1, 3)
col = torch.tensor([[1.0], [2.0], [3.0], [4.0]])  # shape (4, 1)

print("M + row:\n", M + row)                  # adds row to every row
print("M + col:\n", M + col)                  # adds col to every column
print("row + col shape:", (row + col).shape)   # (1,3) and (4,1) broadcast to (4,3)

try:
    M + torch.ones(4)    # (4,3) and (4,) -> trailing dims 3 vs 4 don't match
except RuntimeError as err:
    print("\nError, as expected:", str(err)[:80])

## 4. Matrix multiplication

`@` is matrix multiplication. It's almost all of the compute in a neural network. The rule: `(n, k) @ (k, m) -> (n, m)`. The inner dimensions must match.

In [ ]:
X = torch.randn(32, 784)     # a batch of 32 flattened 28x28 images
W = torch.randn(784, 128)    # weights of a layer mapping 784 inputs to 128 outputs
b = torch.randn(128)

out = X @ W + b              # this one line IS a fully connected layer
print(out.shape)

# Batched matmul: leading dims are treated as a batch. This is what attention does.
Q = torch.randn(8, 16, 64)   # (batch, seq, dim)
K = torch.randn(8, 16, 64)
scores = Q @ K.transpose(-2, -1)   # (8,16,64) @ (8,64,16) -> (8,16,16)
print(scores.shape)

That `scores` line is the core of self-attention in a transformer. Every token computes a similarity score with every other token. You'll build the rest of it in module 06.

### What one entry of a matrix product is

The shape rule tells you the size of the output. This tells you what goes in each slot. Write $Y = XW$, where $X$ has shape $(n, K)$ and $W$ has shape $(K, m)$. Then

$$Y_{ij} = \sum_{k=1}^{K} X_{ik}\, W_{kj}$$

In words: row $i$, column $j$ of the output is row $i$ of $X$ dotted with column $j$ of $W$. Multiply the two lists of $K$ numbers pairwise and add up the products. Here $i$ picks a row of $X$ (which example in the batch), $j$ picks a column of $W$ (which output feature), and $k$ walks along the shared inner dimension, which is why the inner sizes have to match. For the layer above, `out[i, j]` is a weighted sum of image $i$'s 784 pixels, with column $j$ of `W` as the weights.

The transpose of a matrix, written $A^\top$ (`A.T` in PyTorch), flips it over its diagonal: rows become columns, so $(A^\top)_{ij} = A_{ji}$ and shape $(n, m)$ becomes $(m, n)$. You reach for a transpose when the inner dimensions don't line up yet. That's what `K.transpose(-2, -1)` did above: it turned each $(16, 64)$ matrix into $(64, 16)$ so that every query could be dotted with every key. Module 03 uses both facts to work out how gradients flow backward through a matrix multiply.

In [ ]:
# One entry of X @ W by hand: row i of X dotted with column j of W.
i, j = 3, 7
by_hand = sum(X[i, k] * W[k, j] for k in range(X.shape[1]))
print("by hand:      ", by_hand.item())
print("(X @ W)[i, j]:", (X @ W)[i, j].item())
print("match:", torch.allclose(by_hand, (X @ W)[i, j], atol=1e-4))

# Transpose: rows become columns, and A.T[i, j] == A[j, i].
A = torch.arange(6).reshape(2, 3)
print(A, tuple(A.shape))
print(A.T, tuple(A.T.shape))

The two numbers agree up to float32 rounding (the loop adds the 784 products in a different order than the matmul kernel does). `@` computes all 32 x 128 of those dot products at once, on hardware built for it.

## 5. Using your Mac's GPU

Tensors live on a device, and every tensor in an operation must be on the same device. Move them with `.to(device)`.

GPUs only win when there's enough work to spread across their cores. Watch the crossover.

In [ ]:
def sync():
    # GPU work is asynchronous. Without syncing, the timer would stop before the GPU finished.
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize()

def time_matmul(n, dev, repeats=5):
    A = torch.randn(n, n, device=dev)
    B = torch.randn(n, n, device=dev)
    A @ B; sync()                        # warm-up run, not timed
    start = time.perf_counter()
    for _ in range(repeats):
        A @ B
    sync()
    return (time.perf_counter() - start) / repeats

for n in [64, 512, 2048, 4096]:
    cpu_t = time_matmul(n, torch.device("cpu"))
    line = f"n={n:5}  cpu {cpu_t*1000:8.2f} ms"
    if device.type != "cpu":
        gpu_t = time_matmul(n, device)
        line += f"   {device.type} {gpu_t*1000:8.2f} ms   speedup {cpu_t/gpu_t:5.1f}x"
    print(line)

In [ ]:
cpu_tensor = torch.ones(3)
gpu_tensor = torch.ones(3, device=device)

try:
    cpu_tensor + gpu_tensor
    print("Same device (you're on CPU), so no error here.")
except RuntimeError as err:
    print("Error:", str(err)[:100])

print(cpu_tensor.to(device) + gpu_tensor)

Open Activity Monitor > Window > GPU History and rerun the benchmark cell with `n=4096` to watch your GPU spike.

## 6. Autograd

Training means nudging every weight in the direction that lowers the loss. To do that you need the gradient: how much the loss changes when each weight changes a tiny bit. Autograd computes all of those for you.

Take `y = x² + 3x`. By calculus, `dy/dx = 2x + 3`, so at `x = 2` the gradient is 7.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)   # "track operations on x"
y = x**2 + 3*x
y.backward()                                # compute dy/dx and store it in x.grad
print("y =", y.item(), "  dy/dx =", x.grad.item())

Three behaviors that cause real bugs:

In [ ]:
# 1. Gradients ACCUMULATE. Calling backward twice adds, it doesn't overwrite.
x = torch.tensor(2.0, requires_grad=True)
(x**2 + 3*x).backward()
(x**2 + 3*x).backward()
print("After two backward calls:", x.grad.item(), "(7 + 7)")
x.grad.zero_()
print("After zero_():", x.grad.item())

# 2. Inside torch.no_grad(), nothing is tracked. Use it for inference and weight updates.
with torch.no_grad():
    z = x * 10
print("z.requires_grad:", z.requires_grad)

# 3. .detach() gives a tensor cut off from the graph. .item() gives a plain Python number.
print(type(y.detach()), type(y.item()))

## 7. Gradient descent by hand

Now use gradients to learn. You'll fit a line `y = w*x + b` to noisy data, with no `nn` module and no optimizer, so you can see every step. The true values are `w = 2.5`, `b = -1.0`. The model starts knowing nothing.

In [ ]:
torch.manual_seed(0)
X = torch.linspace(-3, 3, 100)
Y = 2.5 * X - 1.0 + 0.5 * torch.randn(100)     # true line plus noise

w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.05

for step in range(101):
    pred = w * X + b                           # 1. forward pass
    loss = ((pred - Y) ** 2).mean()            # 2. loss (mean squared error)
    loss.backward()                            # 3. backward pass: compute d(loss)/dw and d(loss)/db
    with torch.no_grad():                      # 4. update: step downhill
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()             # 5. reset gradients for the next step
    if step % 20 == 0:
        print(f"step {step:3}  loss {loss.item():.4f}  w {w.item():.3f}  b {b.item():.3f}")

Those five numbered steps are training. Every model you'll ever train, including GPT-scale ones, runs this same loop. The difference is just a bigger model, more data, and a smarter update rule.

## 8. The same thing, the PyTorch way

In practice you use `nn.Module` for the model, a loss function from `nn`, and an optimizer that handles the update and zeroing.

In [ ]:
import torch.nn as nn

model = nn.Linear(in_features=1, out_features=1)          # holds w and b for you
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

Xc, Yc = X.unsqueeze(1), Y.unsqueeze(1)   # nn layers expect (batch, features): (100,) -> (100, 1)

for step in range(101):
    pred = model(Xc)
    loss = loss_fn(pred, Yc)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print(f"step {step:3}  loss {loss.item():.4f}")

print("Learned w:", model.weight.item(), " b:", model.bias.item())

## 9. The geometry of gradient descent

Sections 7 and 8 told you to "step downhill" by subtracting `lr * grad`. This section shows why that direction is the right one, how big the step is allowed to be, and why plain gradient descent struggles on some loss surfaces. Sections 10 and 11 build on it.

### The gradient is a vector

Collect all the parameters into one vector $\theta = (\theta_1, \dots, \theta_n)$. In section 7, $\theta = (w, b)$, so $n = 2$. The loss $L(\theta)$ is a single number. The gradient is the vector of its partial derivatives:

$$\nabla L(\theta) = \left(\frac{\partial L}{\partial \theta_1}, \dots, \frac{\partial L}{\partial \theta_n}\right)$$

$\nabla$ ("nabla") means "gradient of". Each partial derivative $\partial L / \partial \theta_i$ is how fast the loss changes when you nudge only $\theta_i$ and hold every other parameter fixed. That's exactly what `w.grad` and `b.grad` held in section 7.

### Why the negative gradient is the steepest way down

Suppose you take a tiny step in some direction $u$. Let $u$ be a unit vector, meaning its length $\|u\| = \sqrt{u_1^2 + \dots + u_n^2}$ equals 1, so it encodes a direction and nothing else. The rate at which $L$ changes as you move along $u$ is the directional derivative:

$$D_u L = \nabla L \cdot u = \sum_{i=1}^{n} \frac{\partial L}{\partial \theta_i}\, u_i$$

It's a dot product: each parameter's sensitivity times how much the direction moves that parameter, summed up. A dot product also has a geometric form. For any two vectors $a$ and $b$, $a \cdot b = \|a\|\,\|b\|\cos\varphi$, where $\varphi$ is the angle between them. (The angle gets $\varphi$ because $\theta$ already means the parameters.) Since $\|u\| = 1$:

$$D_u L = \|\nabla L\| \cos\varphi$$

$\cos\varphi$ runs from $-1$ to $1$, so:

- At $\varphi = 0$ (step along the gradient), $D_u L = +\|\nabla L\|$, the fastest possible increase. The gradient points in the direction of steepest ascent, and its length says how steep that is.
- At $\varphi = 180^\circ$ (step against the gradient), $D_u L = -\|\nabla L\|$, the fastest possible decrease. That's why the update is $\theta - \eta \nabla L$, with a minus sign.
- At $\varphi = 90^\circ$, $D_u L = 0$. Moving perpendicular to the gradient doesn't change the loss at all, to first order. Those directions trace a contour line (a curve of constant loss), so the gradient always crosses contour lines at right angles.

### Why a small enough step always helps

The first-order Taylor approximation says that close to $\theta$, a smooth function looks like its tangent plane: $L(\theta + \Delta) \approx L(\theta) + \nabla L \cdot \Delta$ for a small step $\Delta$. Gradient descent uses $\Delta = -\eta \nabla L$, where $\eta$ ("eta") is the learning rate, `lr` in the code. Plug it in:

$$L(\theta - \eta \nabla L) \approx L(\theta) - \eta\, \nabla L \cdot \nabla L = L(\theta) - \eta\, \|\nabla L\|^2$$

$\|\nabla L\|^2$ is a sum of squares, so it's positive whenever the gradient isn't zero, and the predicted change is always a decrease. The catch is the "approximately". The approximation ignores curvature (the terms in $\eta^2$ of a fuller Taylor expansion), and those take over when $\eta$ is large. A small enough step is guaranteed to lower the loss. Too big a step can overshoot. You'll see both below.

### Seeing it on section 7's loss surface

Section 7's loss is a function of two numbers:

$$L(w, b) = \frac{1}{N}\sum_{i=1}^{N} \left(w x_i + b - y_i\right)^2$$

the mean squared error over the $N = 100$ points $(x_i, y_i)$. Two inputs means you can draw it as a contour map, like a topographic map of a valley. The next cell rebuilds section 7's data with the same seed (section 7's `w` and `b` are already trained, so it doesn't reuse them) and reruns section 7's loop as a function that records every $(w, b)$ it visits.

In [ ]:
import matplotlib.pyplot as plt

# Rebuild section 7's data and starting point: same seed, same calls, same order,
# so torch.randn produces the same noise and the same initial w.
torch.manual_seed(0)
x_line = torch.linspace(-3, 3, 100)
y_line = 2.5 * x_line - 1.0 + 0.5 * torch.randn(100)
w_start, b_start = torch.randn(1).item(), 0.0

def line_loss(w, b):
    # Works for single values and for whole grids: [..., None] lines w and b up against the 100 points.
    pred = w[..., None] * x_line + b[..., None]
    return ((pred - y_line) ** 2).mean(dim=-1)

def descend_line(lr, steps=101):
    # Section 7's five-step loop, returning every (w, b) it visits.
    w = torch.tensor(w_start, requires_grad=True)
    b = torch.tensor(b_start, requires_grad=True)
    path = [(w_start, b_start)]
    for _ in range(steps):
        loss = line_loss(w, b)
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_(); b.grad.zero_()
        path.append((w.item(), b.item()))
    return torch.tensor(path)

line_path = descend_line(lr=0.05)
print(f"start (w, b) = ({line_path[0, 0]:.3f}, {line_path[0, 1]:.3f})")
print(f"end   (w, b) = ({line_path[-1, 0]:.3f}, {line_path[-1, 1]:.3f})   <- same as section 7's last line")

Now the picture. The cell evaluates the loss on a 200 x 200 grid of $(w, b)$ values (broadcasting does all 40,000 at once), draws its contours, overlays the path, and adds red arrows showing the negative gradient at a grid of points. The arrows all have the same length because only their direction matters here. The axes use equal scaling, so a right angle on screen is a real right angle.

In [ ]:
w_grid, b_grid = torch.meshgrid(torch.linspace(-0.5, 4.0, 200), torch.linspace(-3.0, 1.5, 200), indexing="xy")
loss_grid = line_loss(w_grid, b_grid)

# Gradients at a 6 x 6 grid of points. Each point's loss depends only on its own (w, b),
# so one backward pass on the sum hands every point its own gradient.
wq, bq = torch.meshgrid(torch.linspace(0.0, 3.5, 6), torch.linspace(-2.5, 1.0, 6), indexing="xy")
wq, bq = wq.clone().requires_grad_(), bq.clone().requires_grad_()
line_loss(wq, bq).sum().backward()
norm = torch.sqrt(wq.grad**2 + bq.grad**2)
arrow_len = 0.35
dw, db = -arrow_len * wq.grad / norm, -arrow_len * bq.grad / norm

fig, ax = plt.subplots(figsize=(7.5, 7.5))
cs = ax.contour(w_grid.numpy(), b_grid.numpy(), loss_grid.numpy(),
                levels=torch.logspace(-0.5, 1.6, 12).tolist(), cmap="viridis")
ax.clabel(cs, fmt="%.2g", fontsize=8)
ax.quiver(wq.detach().numpy(), bq.detach().numpy(), dw.numpy(), db.numpy(), angles="xy", scale_units="xy",
          scale=1, color="tab:red", width=0.004, label="negative gradient (direction only)")
ax.plot(line_path[:, 0], line_path[:, 1], "o-", ms=3, color="black", label="gradient descent path, lr = 0.05")
ax.plot(2.5, -1.0, "*", ms=16, color="gold", mec="black", label="true (w, b) = (2.5, -1.0)")
ax.set_aspect("equal")
ax.set_xlabel("w"); ax.set_ylabel("b")
ax.set_title("Section 7's loss surface L(w, b), contours on a log scale")
ax.legend(loc="upper left", fontsize=8)
plt.show()

Every red arrow crosses its contour line at a right angle, as the $\cos\varphi$ argument predicted. The contours are ellipses stretched along $b$, so "perpendicular to the contour" isn't the same as "toward the minimum": the arrows aim at the nearest part of the valley floor. That's why the path doesn't head straight for the star. It moves mostly in $w$ first, then turns and slides along the valley in $b$. The ellipses are stretched because the loss curves more sharply in $w$ than in $b$. Changing $w$ shifts each prediction by $x_i$ (up to 3 in size), while changing $b$ shifts every prediction by exactly 1.

### How big can the step be?

Take the simplest possible loss, a one-parameter bowl:

$$L(w) = \frac{\lambda}{2} w^2$$

$\lambda$ ("lambda") is the curvature: the second derivative $L''(w) = \lambda$, which measures how fast the slope itself changes. Large $\lambda$ means a steep, narrow bowl. The gradient is $L'(w) = \lambda w$, so one gradient descent step is

$$w \leftarrow w - \eta \lambda w = (1 - \eta\lambda)\, w$$

Every step multiplies $w$ by the same factor $r = 1 - \eta\lambda$, so after $t$ steps $w_t = r^t w_0$. Everything depends on $r$:

- $0 < \eta < 1/\lambda$ gives $0 < r < 1$. $w$ shrinks toward 0 smoothly, from one side.
- $\eta = 1/\lambda$ gives $r = 0$. One step lands exactly on the minimum.
- $1/\lambda < \eta < 2/\lambda$ gives $-1 < r < 0$. $w$ flips sign every step (it overshoots the bottom) but still shrinks.
- $\eta = 2/\lambda$ gives $r = -1$. $w$ bounces between $+w_0$ and $-w_0$ forever.
- $\eta > 2/\lambda$ gives $|r| > 1$. Each overshoot is bigger than the last, and $w$ blows up.

So gradient descent converges only if $0 < \eta < 2/\lambda$. The steeper the bowl, the smaller the learning rate has to be.

The cell below uses $\lambda = 4$, so $1/\lambda = 0.25$ and $2/\lambda = 0.5$. Before you run it, predict which of the six learning rates converge smoothly, which oscillate, and which diverge.

In [ ]:
lam = 4.0
etas = [0.05, 0.2, 0.4, 0.48, 0.5, 0.55]

fig, ax = plt.subplots(figsize=(9, 4.5))
for eta in etas:
    w = 1.0
    ws = [w]
    for _ in range(25):
        w = w - eta * lam * w            # the gradient of (lam/2) * w^2 is lam * w
        ws.append(w)
    r = 1 - eta * lam
    print(f"eta = {eta:<4}  r = 1 - eta*lam = {r:+.2f}   first steps: {[round(v, 3) for v in ws[:6]]}")
    ax.plot([abs(v) for v in ws], marker=".", label=f"eta = {eta}  (r = {r:+.2f})")
ax.set_yscale("log")
ax.set_xlabel("step"); ax.set_ylabel("|w|  (log scale)")
ax.set_title("Gradient descent on L(w) = (lambda/2) w^2 with lambda = 4")
ax.legend(fontsize=8)
plt.show()

Only $\eta = 0.55$ diverges, and $\eta = 0.5$ sits exactly on the edge, bouncing between $+1$ and $-1$. The two oscillating runs (0.4 and 0.48) flip sign every step, which you can see in the printed values, but still shrink. The fastest is $\eta = 0.2$, the one closest to $1/\lambda$. Both extremes are slow: $\eta = 0.05$ because each step is tiny, and $\eta = 0.48$ because it spends most of every step overshooting. On a log axis, steady shrinking by a factor $|r|$ per step is a straight line with slope $\log|r|$. That's the shape of every learning rate sweep you'll run: too small is slow, too large blows up, and the best value sits somewhat below the blow-up point.

A real loss isn't one bowl, but near a minimum it looks like one in every direction, each with its own curvature. Section 7's line fit is an example. Because its $x$ values are centered on 0, $w$ and $b$ don't interact, and differentiating $L(w, b)$ twice gives a curvature of $\lambda_w = 2 \cdot \text{mean}(x_i^2) \approx 6.1$ in the $w$ direction and $\lambda_b = 2$ in the $b$ direction. The steep $w$ direction sets the limit: $\eta < 2/6.1 \approx 0.33$.

Try this: in a new cell, run `descend_line(0.32)[-1]` and `descend_line(0.34)[-1]`, and predict the results first. One ends near $(2.5, -1.0)$ after zig-zagging in $w$, and the other runs off toward infinity.

### When the bowl is long and narrow

Now put two very different curvatures in one loss:

$$L(w_1, w_2) = \tfrac{1}{2}\left(\lambda_1 w_1^2 + \lambda_2 w_2^2\right), \qquad \lambda_1 = 1,\ \ \lambda_2 = 100$$

Each coordinate is its own 1D bowl: $w_1$ sits in a shallow one and $w_2$ in a steep one. The contours are ellipses $\sqrt{\lambda_2/\lambda_1} = 10$ times longer than they are wide. The ratio of the largest curvature to the smallest, $\lambda_{\max}/\lambda_{\min}$ (100 here), is called the condition number, and a loss with a large one is ill-conditioned.

One learning rate has to serve both directions, and the steep one caps it at $\eta < 2/\lambda_2 = 0.02$. At $\eta = 0.019$, the steep coordinate gets multiplied by $1 - 0.019 \times 100 = -0.9$ per step, so it zig-zags across the valley. The shallow coordinate gets multiplied by $1 - 0.019 \times 1 = 0.981$, so it creeps along the valley floor, losing under 2% of its distance per step.

In [ ]:
bowl_lams = torch.tensor([1.0, 100.0])      # curvature along w1 (shallow) and w2 (steep)
bowl_start = torch.tensor([-5.0, 0.5])

def bowl_loss(w):
    return 0.5 * (bowl_lams * w**2).sum(dim=-1)

def plot_bowl(ax, paths):
    # Contours of the bowl with one or more optimizer paths on top. Section 11 reuses this.
    g1, g2 = torch.meshgrid(torch.linspace(-5.5, 2.0, 300), torch.linspace(-0.8, 0.8, 120), indexing="xy")
    ax.contour(g1.numpy(), g2.numpy(), bowl_loss(torch.stack([g1, g2], dim=-1)).numpy(),
               levels=torch.logspace(-2, 1.5, 12).tolist(), cmap="viridis", alpha=0.7)
    for label, p in paths.items():
        ax.plot(p[:, 0], p[:, 1], ".-", ms=3, lw=1, label=label)
    ax.plot(0, 0, "*", ms=14, color="gold", mec="black", label="minimum")
    ax.set_xlim(-5.5, 2.0); ax.set_ylim(-0.8, 0.8); ax.set_aspect("equal")
    ax.set_xlabel("w1 (shallow direction)"); ax.set_ylabel("w2 (steep direction)")
    ax.legend(loc="lower right", fontsize=8)

eta = 0.019
w = bowl_start.clone()
gd_path = [w.clone()]
for _ in range(100):
    grad = bowl_lams * w                     # gradient of the bowl: (lambda1 * w1, lambda2 * w2)
    w = w - eta * grad
    gd_path.append(w.clone())
gd_path = torch.stack(gd_path)
print(f"start: w = {bowl_start.tolist()}   loss = {bowl_loss(bowl_start):.2f}")
print(f"after 100 steps: w = ({gd_path[-1, 0]:.3f}, {gd_path[-1, 1]:.1e})   loss = {bowl_loss(gd_path[-1]):.3f}")

fig, ax = plt.subplots(figsize=(12, 3.8))
plot_bowl(ax, {f"gradient descent, lr = {eta}": gd_path})
ax.set_title("Ill-conditioned bowl: curvature 1 along w1, 100 along w2")
plt.show()

The first steps bounce back and forth across the narrow direction. The bounce shrinks by 10% per step and fades out over the first few dozen steps. After that the path creeps along the valley floor, and after 100 steps $w_1$ is still about $-0.7$, only 85% of the way from $-5$ to the minimum. The steep direction was solved almost immediately ($w_2$ is down near $10^{-5}$), but it still dictated the learning rate for the whole run.

This is the core weakness of plain gradient descent: the learning rate is set by the steepest direction, and progress is set by the shallowest. Real networks have millions of directions whose curvatures span many orders of magnitude, so this is the normal situation in deep learning. Two fixes follow from it. The first is to make the bowl rounder before you start. Inputs on very different scales create exactly this kind of stretched loss, which is why you normalize features to mean 0 and standard deviation 1 (exercise 2, and module 05 does it to images). The second is an update rule that adapts to the shape of the surface, which is section 11.

Try this: set `eta = 0.021`, just past $2/\lambda_2$, and rerun. The shallow direction would be perfectly happy with that step size, but the steep one diverges and takes the whole run with it.

## 10. Local minima, saddle points, and noisy gradients

The bowls in section 9 curve upward in every direction, everywhere. A function like that is called convex: it has a single valley, and gradient descent with a small enough learning rate reaches its bottom from any start. Neural network losses aren't convex: they have many valleys, ridges, and flat stretches. This section looks at two things that can happen on such a surface, and then drops the other simplification sections 7 to 9 made, which is computing the exact gradient over all the data at every step.

### Different starts, different minima

Take $f(w) = w^4 - 3w^2 + w$. Its slope $f'(w) = 4w^3 - 6w + 1$ is zero at three points: two minima, near $w \approx -1.30$ and $w \approx 1.13$, with a local maximum between them near $w \approx 0.17$. Gradient descent only sees the local slope, so it rolls into whichever valley it starts in. Predict where each of the five starts below ends up.

In [ ]:
def f(w):
    return w**4 - 3 * w**2 + w

fig, ax = plt.subplots(figsize=(9, 4.5))
grid = torch.linspace(-2.1, 2.1, 400)
ax.plot(grid, f(grid), color="gray", lw=2, label="f(w) = w^4 - 3w^2 + w")
for w0 in [-2.0, -0.5, 0.1, 0.3, 2.0]:
    w = torch.tensor(w0, requires_grad=True)
    path = [w0]
    for _ in range(100):
        f(w).backward()
        with torch.no_grad():
            w -= 0.02 * w.grad
        w.grad.zero_()
        path.append(w.item())
    path = torch.tensor(path)
    print(f"start {w0:+.1f}  ->  w = {path[-1]:+.4f}   f(w) = {f(path[-1]):+.4f}")
    ax.plot(path, f(path), "o-", ms=3, lw=1, label=f"start {w0:+.1f}, ends at w = {path[-1]:+.2f}")
ax.set_xlabel("w"); ax.set_ylabel("f(w)")
ax.set_title("Gradient descent (lr = 0.02, 100 steps) on a function with two minima")
ax.legend(fontsize=8)
plt.show()

The starts at $-2.0$, $-0.5$, and $0.1$ end in the left valley, and $0.3$ and $2.0$ end in the right one. The starts $0.1$ and $0.3$ are only 0.2 apart but land in different minima, because they sit on opposite sides of the hump at $w \approx 0.17$. The right minimum ($f \approx -1.07$) is much worse than the left one ($f \approx -3.51$), and gradient descent has no way to know: at the bottom of either valley the gradient is zero and the updates stop. That's a local minimum.

### Saddle points

In two or more dimensions there's a third kind of flat spot. Take $f(x, y) = x^2 - y^2$. Its gradient is $\nabla f = (2x, -2y)$, which is zero at the origin. But the origin isn't a minimum: along $x$ the function curves up (a valley), and along $y$ it curves down (a ridge). It's shaped like a horse saddle, or a mountain pass.

Gradient descent on it is two separate 1D updates: $x \leftarrow (1 - 2\eta)\,x$ shrinks every step, and $y \leftarrow (1 + 2\eta)\,y$ grows every step, unless $y$ is exactly 0, in which case it stays 0 forever. The cell runs two starts with $\eta = 0.1$: one exactly on the line $y = 0$, and one a hair off it at $y = 10^{-3}$.

In [ ]:
def saddle(p):
    return p[0] ** 2 - p[1] ** 2

def descend_saddle(start, lr=0.1, steps=40):
    p = torch.tensor(start, requires_grad=True)
    path, grad_norms = [p.detach().clone()], []
    for _ in range(steps):
        saddle(p).backward()
        grad_norms.append(p.grad.norm().item())
        with torch.no_grad():
            p -= lr * p.grad
        p.grad.zero_()
        path.append(p.detach().clone())
    return torch.stack(path), grad_norms

stuck, stuck_g = descend_saddle([1.0, 0.0])
escape, escape_g = descend_saddle([1.0, 1e-3])
print(f"start (1, 0):      ends at ({stuck[-1, 0]:.1e}, {stuck[-1, 1]:.1e})   last gradient norm {stuck_g[-1]:.1e}")
print(f"start (1, 0.001):  ends at ({escape[-1, 0]:.1e}, {escape[-1, 1]:.2f})   last gradient norm {escape_g[-1]:.2f}")
print(f"smallest gradient norm on the escaping run: {min(escape_g):.3f} (step {escape_g.index(min(escape_g))})")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
gx, gy = torch.meshgrid(torch.linspace(-1.6, 1.6, 200), torch.linspace(-1.6, 1.6, 200), indexing="xy")
ax1.contour(gx.numpy(), gy.numpy(), (gx**2 - gy**2).numpy(), levels=15, cmap="coolwarm")
ax1.plot(escape[:, 0], escape[:, 1], "s-", ms=3, color="tab:orange", label="start (1, 0.001): escapes")
ax1.plot(stuck[:, 0], stuck[:, 1], "o", ms=7, mfc="none", color="tab:blue", label="start (1, 0): stalls at the saddle")
ax1.plot(0, 0, "kx", ms=10, mew=2)
ax1.set_xlim(-1.6, 1.6); ax1.set_ylim(-1.6, 1.6); ax1.set_aspect("equal")
ax1.set_xlabel("x"); ax1.set_ylabel("y"); ax1.set_title("f(x, y) = x^2 - y^2 (x marks the saddle)")
ax1.legend(loc="lower left", fontsize=8)

ax2.plot(escape[:, 1].abs(), label="|y|, start (1, 0.001)")
ax2.plot(escape_g, label="gradient norm, start (1, 0.001)")
ax2.plot(stuck_g, "--", label="gradient norm, start (1, 0)")
ax2.set_yscale("log"); ax2.set_xlabel("step"); ax2.set_ylabel("log scale")
ax2.set_title("Escaping the saddle")
ax2.legend(fontsize=8)
plt.show()

Both runs head for the origin at first, because the $x$ part of the gradient dominates. The run that starts exactly on $y = 0$ never gets any push in $y$, so it parks at the saddle with a gradient norm around $3 \times 10^{-4}$ and still falling, which looks exactly like convergence. The other run's $|y|$ grows by a factor of $1 + 2\eta = 1.2$ every step (a straight line on the log plot), so the $10^{-3}$ offset becomes about 1.5 after 40 steps. Its gradient norm dips to about 0.06 near the saddle, then climbs back up as it slides off. This toy function is unbounded below, so here escaping means sliding downhill forever. In a real loss, the direction that curves down leads somewhere lower, and training carries on from there.

Why care about saddles? At a point where the gradient is zero, each direction curves either up or down. (Formally, those curvatures are the eigenvalues of the Hessian, the matrix of second derivatives. You don't need that machinery here.) A minimum needs every direction to curve up. A network with millions of parameters has millions of directions at every such point, and asking all of them to curve up is a very strong condition. So in high dimensions, most points with zero gradient are saddles, not minima. As a rough intuition, not a theorem: if each direction were a coin flip, the chance that all $n$ of them curve up would be $2^{-n}$. Work on large networks (for example Dauphin et al., 2014) finds that the local minima training actually reaches tend to have losses close to each other and close to the best available, so getting trapped in a bad local minimum is rarely the practical problem. Slow progress near saddles and through flat or ill-conditioned regions is the more common one, and that's what momentum and Adam help with.

Try this: change the second run to `escape, escape_g = descend_saddle([1.0, 1e-8], steps=110)`. It still escapes, just later. Each step multiplies $|y|$ by 1.2, so getting from $10^{-8}$ up to $10^{-3}$ costs $\ln(10^5)/\ln(1.2) \approx 63$ extra steps. Check that against the plot.

### Noisy gradients from mini-batches

Sections 7 to 9 computed the loss over all 100 points at every step. That's full-batch gradient descent. Real datasets are far bigger (60,000 images in module 05, billions of tokens for an LLM), and one exact gradient per step would cost a full pass over the data. Instead, each step draws a random mini-batch of $B$ examples and uses the gradient on just those. That's stochastic gradient descent (SGD), and it's what every training run from module 05 on does.

The full loss is an average over all $N$ examples, $L = \frac{1}{N}\sum_{i=1}^{N} \ell_i$, where $\ell_i$ is the loss on example $i$ alone. Derivatives pass through sums, so the full gradient is also an average, $\nabla L = \frac{1}{N}\sum_{i=1}^{N} \nabla \ell_i$. A mini-batch gradient averages $B$ of those per-example gradients instead of all $N$:

$$g_B = \frac{1}{B}\sum_{i \in \text{batch}} \nabla \ell_i$$

When the batch is drawn at random, $g_B$ is an unbiased estimate of $\nabla L$: averaged over many batches, it equals the full gradient exactly. Any single $g_B$ is off by a random error. How big? Say the per-example gradients scatter around their mean $\nabla L$ with standard deviation $\sigma$. The mean of $B$ independent draws has standard deviation

$$\text{std}(g_B) = \frac{\sigma}{\sqrt{B}}$$

because averaging $B$ independent errors divides their variance by $B$, and the standard deviation is the square root of the variance. So 4 times the batch size buys only 2 times less noise.

The next cell checks this on section 7's line fit, frozen at the point where section 7 started. It computes the full gradient, the 100 per-example gradients (to measure $\sigma$), and 500 random mini-batch gradients for each of $B = 4$, 16, and 64. "Spread" means the root-mean-square distance between a mini-batch gradient and the full gradient. The batches are drawn with replacement (`torch.randint`), so the draws are independent, as the formula assumes. Without replacement, a batch of 64 out of only 100 points would overlap heavily with the full dataset, and the noise would shrink faster than $1/\sqrt{B}$.

In [ ]:
torch.manual_seed(0)
w_fix = torch.tensor(w_start, requires_grad=True)     # section 7's starting point, rebuilt in section 9
b_fix = torch.tensor(b_start, requires_grad=True)

def grad_on(idx):
    # Gradient (dL/dw, dL/db) of the mean squared error on just the examples at positions idx.
    w_fix.grad, b_fix.grad = None, None
    ((w_fix * x_line[idx] + b_fix - y_line[idx]) ** 2).mean().backward()
    return torch.stack([w_fix.grad, b_fix.grad])

N = len(x_line)
full = grad_on(torch.arange(N))
per_example = torch.stack([grad_on(torch.tensor([i])) for i in range(N)])
sigma = (per_example - full).pow(2).sum(dim=1).mean().sqrt()
print(f"full gradient (dL/dw, dL/db) = ({full[0]:.3f}, {full[1]:.3f})")
print(f"mean of the 100 per-example gradients = ({per_example[:, 0].mean():.3f}, {per_example[:, 1].mean():.3f})")
print(f"sigma (per-example spread) = {sigma:.3f}\n")

batch_sizes = [4, 16, 64]
batch_grads, spreads = {}, {}
print("   B   mean of 500 batch gradients   spread   sigma/sqrt(B)   spread / spread(B=4)   sqrt(4/B)")
for B in batch_sizes:
    idx = torch.randint(0, N, (500, B))                # 500 batches of B indices, drawn with replacement
    g = torch.stack([grad_on(row) for row in idx])
    batch_grads[B] = g
    spreads[B] = (g - full).pow(2).sum(dim=1).mean().sqrt()
    print(f"{B:4}      ({g[:, 0].mean():7.3f}, {g[:, 1].mean():6.3f})     {spreads[B]:7.3f}   {sigma / B**0.5:10.3f}"
          f"   {spreads[B] / spreads[batch_sizes[0]]:16.2f}   {(4 / B) ** 0.5:12.2f}")

fig, ax = plt.subplots(figsize=(8, 6))
for B in batch_sizes:                                   # default colors: B = 4 blue, 16 orange, 64 green
    ax.scatter(batch_grads[B][:, 0], batch_grads[B][:, 1], s=8, alpha=0.35, label=f"B = {B}")
ax.plot(full[0], full[1], "*", ms=18, color="black", label="full-batch gradient")
ax.set_xlabel("dL/dw"); ax.set_ylabel("dL/db")
ax.set_title("500 mini-batch gradients per batch size, all at the same (w, b)")
ax.legend()
plt.show()

All three clouds are centered on the black star, and the means of the 500 batch gradients land close to the full gradient: that's what unbiased means. The clouds shrink as $B$ grows, and the measured spread tracks $\sigma/\sqrt{B}$, so each 4x increase in batch size cuts the noise roughly in half (the "spread / spread(B=4)" column against $\sqrt{4/B}$). A single batch of 4 can point well away from the true gradient, and some of them even get the sign of $\partial L/\partial b$ wrong (the blue points below 0). A batch of 64 stays close.

So why not always use the full batch? Because the cost of a step grows with $B$ while its noise only falls like $1/\sqrt{B}$. A batch of 64 costs 16 times the compute of a batch of 4 and is only 4 times less noisy. Early in training even a noisy gradient points roughly downhill, so many cheap noisy steps usually make more progress per unit of compute than a few exact ones. The noise also helps: a run sitting near a saddle gets random kicks off the ridge for free, the same job the $y = 10^{-3}$ offset did in the saddle demo.

Batch size and learning rate interact. Less noise per step means you can afford bigger steps, and a common rule of thumb for SGD is that when you multiply the batch size by $k$, you multiply the learning rate by about $k$ too, usually with a warmup at the start (module 05 covers warmup). On a GPU the practical choice is often the largest batch that keeps the hardware busy, since for small models a batch of 128 takes barely longer than a batch of 8 when the GPU's cores would otherwise sit idle. Module 05 trains with mini-batches of 128.

Try this: add 256 to `batch_sizes` and rerun. The ratio keeps following $\sqrt{4/B}$ even though 256 is more than the 100 points in the dataset, because the batches are drawn with replacement.

## 11. Momentum, Adam, and AdamW

Section 8's `optimizer.step()` and section 12's `torch.optim.Adam` hide the update rule. Here you'll write the three update rules you'll meet most often, run them on section 9's ill-conditioned bowl, and check that your versions match PyTorch's exactly.

Notation for all of them: $\theta_t$ is the parameters after step $t$, $g_t = \nabla L(\theta_{t-1})$ is the gradient computed at step $t$, and $\eta$ is the learning rate. Every operation is elementwise, so each parameter gets updated on its own.

### Plain SGD

$$\theta_t = \theta_{t-1} - \eta\, g_t$$

This is what sections 7 to 9 did. Its weakness is section 9's: one $\eta$ for every direction, capped by the steepest one.

### SGD with momentum

PyTorch's version keeps a running velocity $v$ for every parameter, starting at $v_0 = 0$:

$$v_t = \beta\, v_{t-1} + g_t, \qquad \theta_t = \theta_{t-1} - \eta\, v_t$$

$\beta$ (usually 0.9) is how much of the old velocity survives each step. Unroll the first equation and $v_t = g_t + \beta g_{t-1} + \beta^2 g_{t-2} + \dots$, a sum of all past gradients with older ones faded out. Two things follow:

- If the gradient keeps pointing the same way, the sum grows toward $g\,(1 + \beta + \beta^2 + \dots) = g/(1-\beta)$. The step builds up to about $\eta/(1-\beta)$ times the gradient, which is 10 times plain SGD's step at $\beta = 0.9$.
- If the gradient flips sign every step, as it does across section 9's narrow valley, consecutive terms mostly cancel and the velocity stays small.

So momentum speeds up along the valley floor and damps the zig-zag across it, like a heavy ball rolling downhill. Some textbooks write $v_t = \beta v_{t-1} + (1-\beta)\, g_t$ instead. That only rescales $v$ by a constant, which you can fold into $\eta$.

### Adam

Adam keeps two running averages per parameter, both starting at 0:

$$m_t = \beta_1\, m_{t-1} + (1-\beta_1)\, g_t \qquad\qquad v_t = \beta_2\, v_{t-1} + (1-\beta_2)\, g_t^2$$

$m_t$ is a running average of the gradient (the "first moment"), which plays the role of momentum's velocity. $v_t$ is a running average of the squared gradient (the "second moment"), which tracks how large this parameter's gradients typically are, whatever their sign. The defaults are $\beta_1 = 0.9$, an average over roughly the last 10 steps, and $\beta_2 = 0.999$, roughly the last 1,000.

Because both averages start at 0, they're biased toward 0 early on. After the first step, $m_1 = (1-\beta_1)\, g_1 = 0.1\, g_1$, a tenth of the actual gradient. Dividing by $1 - \beta^t$ fixes that:

$$\hat m_t = \frac{m_t}{1 - \beta_1^t} \qquad\qquad \hat v_t = \frac{v_t}{1 - \beta_2^t}$$

At $t = 1$, $\hat m_1 = 0.1\, g_1 / 0.1 = g_1$ exactly. As $t$ grows, $\beta^t \to 0$ and the correction fades away. Then the update:

$$\theta_t = \theta_{t-1} - \eta\, \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}$$

$\epsilon = 10^{-8}$ is only there to avoid dividing by zero. The important part is dividing by $\sqrt{\hat v_t}$, the typical gradient size for that parameter. If a parameter's gradients are consistently around 100, then $\hat m \approx 100$ and $\sqrt{\hat v} \approx 100$, so the step is about $\eta$. If they're consistently around 0.01, the step is still about $\eta$. Every parameter effectively gets its own learning rate, scaled so its steps are about $\eta$ in size no matter how large its gradients are. That's what an ill-conditioned bowl needs: steps in the steep direction get shrunk and steps in the shallow direction get stretched. When a parameter's gradient keeps changing sign, $\hat m$ averages toward 0 while $\sqrt{\hat v}$ stays large, so the step shrinks and noisy directions move cautiously.

### AdamW

Weight decay pulls every weight a little toward 0 each step, which discourages large weights and helps a model generalize (module 05 uses it against overfitting). The classic way to get it is L2 regularization: add $\frac{\lambda}{2}\|\theta\|^2$ to the loss, which adds $\lambda\theta$ to every gradient. Here $\lambda$ is the decay strength (`weight_decay` in PyTorch), not a curvature. With plain SGD, that's the same thing as shrinking the weights. With Adam it isn't: the extra $\lambda\theta$ goes into $g_t$ and then gets divided by $\sqrt{\hat v_t}$, so parameters with large gradients barely get decayed at all. AdamW (Loshchilov and Hutter, 2019) decouples the two. It applies the decay straight to the weights, outside the adaptive scaling,

$$\theta \leftarrow \theta - \eta\lambda\,\theta$$

and then takes a normal Adam step with the unmodified gradient. Every weight shrinks by the same fraction $\eta\lambda$ per step. In PyTorch, `torch.optim.Adam(..., weight_decay=...)` does the L2 version and `torch.optim.AdamW` does the decoupled one. AdamW is the standard optimizer for transformers, and it's the one module 05 trains with.

### The memory bill

SGD stores nothing beyond the parameters and their gradients. Momentum adds one number per parameter ($v$). Adam and AdamW add two ($m$ and $v$). For a 7B-parameter model in float32, that's 28 GB of weights, 28 GB of gradients, and 56 GB of Adam state before a single activation is stored. Module 07 comes back to this to explain why full fine-tuning is so memory-hungry and why LoRA trains only a small fraction of the parameters.

The next cell implements all three as small classes with the same interface as `torch.optim`: they hold their own state and update the parameters in `step()`.

In [ ]:
class MySGD:
    def __init__(self, params, lr):
        self.params, self.lr = list(params), lr

    @torch.no_grad()
    def step(self):
        for p in self.params:
            p -= self.lr * p.grad


class MyMomentum:
    def __init__(self, params, lr, beta=0.9):
        self.params, self.lr, self.beta = list(params), lr, beta
        self.v = [torch.zeros_like(p) for p in self.params]      # one velocity per parameter, starting at 0

    @torch.no_grad()
    def step(self):
        for p, v in zip(self.params, self.v):
            v.mul_(self.beta).add_(p.grad)                        # v = beta * v + g, in place so the state persists
            p -= self.lr * v


class MyAdam:
    def __init__(self, params, lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        self.params, self.lr, self.eps, self.weight_decay = list(params), lr, eps, weight_decay
        self.beta1, self.beta2 = betas
        self.m = [torch.zeros_like(p) for p in self.params]      # running average of g
        self.v = [torch.zeros_like(p) for p in self.params]      # running average of g^2
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            g = p.grad
            if self.weight_decay:
                p.mul_(1 - self.lr * self.weight_decay)          # AdamW: shrink the weights directly, outside the adaptive scaling
            m.mul_(self.beta1).add_((1 - self.beta1) * g)
            v.mul_(self.beta2).add_((1 - self.beta2) * g * g)
            m_hat = m / (1 - self.beta1 ** self.t)               # bias correction, because m and v started at 0
            v_hat = v / (1 - self.beta2 ** self.t)
            p -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

Now race them on section 9's bowl, all from the same start. SGD keeps section 9's learning rate, 0.019, just under its $2/\lambda_2 = 0.02$ limit. Momentum gets a smaller one, 0.004, because its velocity can build up to $1/(1-\beta) = 10$ times the gradient, so its effective step along a steady direction is 0.04, about twice SGD's. Adam's learning rate means something different (roughly the step size in parameter units), so it gets its own value, 0.1. Before you run it, predict which one finishes with the lowest loss after 150 steps, and what Adam's very first step will be.

In [ ]:
def run_on_bowl(make_opt, steps=150):
    # Optimize section 9's bowl from bowl_start. Returns the path and the loss after every step.
    w = bowl_start.clone().requires_grad_()
    opt = make_opt([w])
    path, losses = [w.detach().clone()], [bowl_loss(w).item()]
    for _ in range(steps):
        loss = bowl_loss(w)
        w.grad = None
        loss.backward()
        opt.step()
        path.append(w.detach().clone())
        losses.append(bowl_loss(w).item())
    return torch.stack(path), losses

runs = {
    "SGD, lr = 0.019": run_on_bowl(lambda p: MySGD(p, lr=0.019)),
    "momentum 0.9, lr = 0.004": run_on_bowl(lambda p: MyMomentum(p, lr=0.004, beta=0.9)),
    "Adam, lr = 0.1": run_on_bowl(lambda p: MyAdam(p, lr=0.1)),
}
print("loss after step:            10         50        100        150")
for name, (path, losses) in runs.items():
    print(f"{name:25} {losses[10]:9.2e}  {losses[50]:9.2e}  {losses[100]:9.2e}  {losses[150]:9.2e}")

adam_path = runs["Adam, lr = 0.1"][0]
print("\ngradient at the start:", (bowl_lams * bowl_start).tolist())
print("Adam's first step:    ", [round(v, 6) for v in (adam_path[1] - adam_path[0]).tolist()])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8.5), gridspec_kw={"height_ratios": [1, 1.1]})
plot_bowl(ax1, {name: path for name, (path, _) in runs.items()})
ax1.set_title("Three optimizers on section 9's bowl, same start")
for name, (_, losses) in runs.items():
    ax2.plot(losses, label=name)
ax2.set_yscale("log"); ax2.set_xlabel("step"); ax2.set_ylabel("loss (log scale)")
ax2.set_title("Loss per step")
ax2.legend()
plt.tight_layout()
plt.show()

Look at Adam's first step first: $(+0.1, -0.1)$, exactly $\eta$ in each coordinate, even though the gradient there was $(-5, 50)$, ten times larger along $w_2$. That's the division by $\sqrt{\hat v}$ at work, and at $t = 1$ it reduces to $-\eta\,\text{sign}(g_1)$: a step of exactly $\eta$ against the gradient in every coordinate. For the first 20 steps Adam moves $w_1$ at close to 0.1 per step (from $-5$ to about $-3.1$), while along the steep $w_2$ direction it moves in a smooth curve instead of bouncing every step.

What the race shows:

- SGD is still creeping after 150 steps (loss about 0.04). Its loss curve is a straight line on the log plot, because the shallow direction loses the same 1.9% of its distance every step.
- Momentum ends lowest at step 150 (loss $5.5 \times 10^{-8}$), but that's partly timing: $w_1$ happens to cross 0 at step 149, and at step 148 its loss was still higher than Adam's. The step-to-step zig-zag is gone, replaced by a slow, smooth wave across the valley that dies out, while the velocity carries it along the valley floor quickly. Its loss curve is bumpy because a heavy ball overshoots: it trades height for speed and climbs a little on the far side of each swing.
- Adam lands close behind (loss around $10^{-6}$), on the smoothest path of the three. Its $m$ is a momentum term too, so it also overshoots. The sharp dip in its loss around step 86 is the moment both coordinates pass through 0 almost together. The momentum in $m$ carries them past, so the loss climbs back up before falling again ($w_1$ stays on the positive side until about step 146).

Don't read this as "momentum beats Adam". Both needed a learning rate chosen for this problem, and on a two-parameter quadratic a tuned momentum is hard to beat. Adam's advantage shows up in real networks, where millions of parameters have gradients that differ in size by orders of magnitude and no single hand-tuned learning rate suits all of them. That's why Adam and AdamW are the defaults for transformers.

Try this: change momentum's `beta` to 0.5 and to 0.99, and Adam's `lr` to 0.01 and to 0.5. Rerun and see how sensitive each optimizer is to its settings.

Last step: prove your classes compute exactly what PyTorch's optimizers compute. The cell runs each pair side by side on a small least-squares problem, from the same starting parameters, for 25 steps, in float64 on the CPU so that any mismatch would be a real difference and not rounding noise. One thing to match: PyTorch's AdamW multiplies the parameters by $(1 - \eta\lambda)$ before the Adam step, which is what `MyAdam` does when `weight_decay` is set. The last row is there for contrast. It compares `torch.optim.Adam` with L2-style `weight_decay` against `torch.optim.AdamW`, which should disagree.

In [ ]:
torch.manual_seed(0)
A = torch.randn(20, 5, dtype=torch.float64)
y = torch.randn(20, dtype=torch.float64)
theta0 = torch.randn(5, dtype=torch.float64)

def compare(make_a, make_b, steps=25):
    # Run two optimizers side by side from theta0. Returns the largest gap seen and whether they end allclose.
    pa, pb = theta0.clone().requires_grad_(), theta0.clone().requires_grad_()
    opt_a, opt_b = make_a([pa]), make_b([pb])
    gap = 0.0
    for _ in range(steps):
        for p, opt in [(pa, opt_a), (pb, opt_b)]:
            p.grad = None
            ((A @ p - y) ** 2).mean().backward()
            opt.step()
        gap = max(gap, (pa - pb).abs().max().item())
    return gap, torch.allclose(pa, pb)

pairs = {
    "MyMomentum vs torch.optim.SGD(momentum=0.9)": (
        lambda p: MyMomentum(p, lr=0.05, beta=0.9), lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
    "MyAdam     vs torch.optim.Adam": (
        lambda p: MyAdam(p, lr=0.05), lambda p: torch.optim.Adam(p, lr=0.05)),
    "MyAdam(wd) vs torch.optim.AdamW(weight_decay=0.1)": (
        lambda p: MyAdam(p, lr=0.05, weight_decay=0.1), lambda p: torch.optim.AdamW(p, lr=0.05, weight_decay=0.1)),
    "contrast: Adam(weight_decay=0.1) vs AdamW(weight_decay=0.1)": (
        lambda p: torch.optim.Adam(p, lr=0.05, weight_decay=0.1), lambda p: torch.optim.AdamW(p, lr=0.05, weight_decay=0.1)),
}
for name, (make_a, make_b) in pairs.items():
    gap, same = compare(make_a, make_b)
    print(f"{name:60} allclose: {str(same):5}  max difference: {gap:.1e}")

All three checks pass, with differences around $10^{-16}$, which is float64 rounding. Your classes compute the same updates as PyTorch's optimizers. The contrast row shows that Adam with L2 weight decay and AdamW really are different algorithms: over 25 steps from the same start, their parameters drift up to about 0.08 apart.

That's what `opt.step()` does. When module 05 calls `torch.optim.AdamW(model.parameters(), lr=..., weight_decay=...)`, PyTorch keeps an $m$ and a $v$ for every weight in the model, divides each step by $\sqrt{\hat v}$, and shrinks every weight by $\eta\lambda$ per step, exactly like `MyAdam`.

## 12. A real (small) neural network

A straight line can't fit a curve. Stack linear layers with a nonlinearity between them and the network can fit almost anything. Here it learns `sin(x)`, and it trains on your GPU with Adam from section 11.

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(0)
X = torch.linspace(-6, 6, 512).unsqueeze(1).to(device)
Y = torch.sin(X)

net = nn.Sequential(
    nn.Linear(1, 64),
    nn.Tanh(),
    nn.Linear(64, 64),
    nn.Tanh(),
    nn.Linear(64, 1),
).to(device)

print(net)
print("Parameters:", sum(p.numel() for p in net.parameters()))

opt = torch.optim.Adam(net.parameters(), lr=1e-2)   # Adam from section 11: per-parameter step sizes, the default for most deep learning
losses = []
for step in range(1500):
    loss = nn.functional.mse_loss(net(X), Y)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(losses); ax1.set_yscale("log"); ax1.set_title("Training loss"); ax1.set_xlabel("step")
with torch.no_grad():
    ax2.plot(X.cpu(), Y.cpu(), label="sin(x)")
    ax2.plot(X.cpu(), net(X).cpu(), "--", label="network")
ax2.legend(); ax2.set_title("Fit")
plt.show()

Try this: delete both `nn.Tanh()` lines and rerun. Without nonlinearities, stacked linear layers collapse into one linear function, so the network can only draw a straight line. That's why activation functions exist.

## Wrap-up

Everything later in this course, and any model you'll ever put behind a product, runs on what's in this notebook: tensors whose shapes you can reason about, autograd for the gradients, and an optimizer stepping downhill. When a training run is slow, unstable, or uses more GPU memory than you budgeted, the cause usually traces back to something here: a learning rate past the $2/\lambda$ edge, a stretched loss surface from unnormalized inputs, a batch size that's too noisy or too big for the hardware, or optimizer state you forgot to count. Knowing these mechanics lets you diagnose those problems yourself instead of guessing at hyperparameters while a rented GPU bills by the hour.

You're done when you can do all five exercises without looking at the solutions, explain from memory what each of the five lines in a training loop does, and explain why the negative gradient is the steepest way down and what Adam's two running averages do.

Going further:

- The official "Learn the Basics" tutorial at pytorch.org/tutorials, for when you want more of the PyTorch API.
- "Why Momentum Really Works" by Gabriel Goh (Distill, 2017), an interactive article on momentum and ill-conditioning.
- The two papers behind section 11, "Adam: A Method for Stochastic Optimization" (Kingma and Ba, 2015) and "Decoupled Weight Decay Regularization" (Loshchilov and Hutter, 2019). You can now follow the update rules in both.
- Next, module 03 builds the autograd engine you used here from scratch, including backprop through matrix multiplies.

---
# Exercises

**Exercise 1: Shapes.** Create `t = torch.arange(24)`. Without running code first, predict the shape of each, then check:
- `t.reshape(2, 3, 4).permute(1, 2, 0)`
- `t.reshape(4, -1)[1:3]`
- `t.reshape(2, 3, 4)[:, None, :, 0]` (`None` inserts a new axis)

**Exercise 2: Normalize columns.** Given `data = torch.randn(100, 5) * 10 + 3`, use broadcasting (no loops) to make every column have mean 0 and standard deviation 1. Verify with `.mean(dim=0)` and `.std(dim=0)`.

**Exercise 3: Softmax from scratch.** Softmax turns a vector of scores into probabilities: `exp(x_i) / sum(exp(x_j))`. Write `my_softmax(x)` that works on the last dimension of any shape, and check it against `torch.softmax(x, dim=-1)`. Then try `torch.tensor([1000.0, 1001.0, 1002.0])`. Your version probably returns `nan`. Fix it by subtracting the max first, and explain to yourself why that doesn't change the answer.

**Exercise 4: Autograd check.** For `f(x) = x³ - 4x` at `x = 3`, compute the derivative by hand, then confirm with autograd.

**Exercise 5: Fit a curve by hand.** Data: `X = torch.linspace(-2, 2, 200)`, `Y = 3*X**2 - 2*X + 1 + 0.3*torch.randn(200)`. Use raw tensors `a, b, c` with `requires_grad=True` and the manual 5-step loop from section 7 to recover the coefficients.

---
# Solutions

In [ ]:
# Exercise 1
t = torch.arange(24)
print(t.reshape(2, 3, 4).permute(1, 2, 0).shape)   # (3, 4, 2)
print(t.reshape(4, -1)[1:3].shape)                 # (2, 6)
print(t.reshape(2, 3, 4)[:, None, :, 0].shape)     # (2, 1, 3)

In [ ]:
# Exercise 2
data = torch.randn(100, 5) * 10 + 3
normalized = (data - data.mean(dim=0)) / data.std(dim=0)   # (100,5) op (5,) broadcasts across rows
print(normalized.mean(dim=0))
print(normalized.std(dim=0))

In [ ]:
# Exercise 3
def my_softmax_naive(x):
    e = torch.exp(x)
    return e / e.sum(dim=-1, keepdim=True)   # keepdim keeps shape (..., 1) so broadcasting lines up

def my_softmax(x):
    x = x - x.max(dim=-1, keepdim=True).values   # exp(x - c) / sum(exp(x - c)) = exp(x) / sum(exp(x))
    e = torch.exp(x)
    return e / e.sum(dim=-1, keepdim=True)

x = torch.randn(4, 10)
print(torch.allclose(my_softmax(x), torch.softmax(x, dim=-1)))

big = torch.tensor([1000.0, 1001.0, 1002.0])
print("naive:", my_softmax_naive(big))   # exp(1000) overflows float32 -> inf/inf = nan
print("stable:", my_softmax(big))

In [ ]:
# Exercise 4: f'(x) = 3x^2 - 4 -> at x = 3: 27 - 4 = 23
x = torch.tensor(3.0, requires_grad=True)
(x**3 - 4*x).backward()
print(x.grad.item())

In [ ]:
# Exercise 5
torch.manual_seed(0)
X = torch.linspace(-2, 2, 200)
Y = 3*X**2 - 2*X + 1 + 0.3*torch.randn(200)

a = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
c = torch.zeros(1, requires_grad=True)
lr = 0.05

for step in range(2001):
    loss = ((a*X**2 + b*X + c - Y) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        for p in (a, b, c):
            p -= lr * p.grad
            p.grad.zero_()
    if step % 500 == 0:
        print(f"step {step:4}  loss {loss.item():.4f}  a {a.item():.3f}  b {b.item():.3f}  c {c.item():.3f}")